# 🐻 Bear Detector — Google Colab Notebook

End-to-end pipeline: **detection → tracking → segmentation → evaluation**

| Step | Description |
|------|-------------|
| 1 | Setup: clone repo, install deps, check GPU |
| 2 | Load configuration |
| 3 | Train YOLOv8 detection model |
| 4 | Detect bears in a sample image |
| 5 | Detect + track bears in a video |
| 6 | Train and run segmentation |
| 7 | Evaluate: mAP, PR curve, MOTA |
| 8 | Browse MLflow experiment logs |

> **GPU recommended.** Go to `Runtime → Change runtime type → T4 GPU` before running.


## 1 · Setup

In [ ]:
# Clone or update the repository and switch to the active development branch
import os

REPO   = 'Bear-Detector'
BRANCH = 'claude/improve-bear-detection-tKY3Y'

if os.path.isfile('pyproject.toml') or os.path.isdir('.git'):
    print('Already inside repo — fetching latest changes...')
    os.system(f'git fetch origin')
    os.system(f'git checkout {BRANCH}')
    os.system(f'git pull --ff-only')
elif os.path.isdir(REPO):
    print('Repo found locally — fetching latest changes...')
    os.system(f'git -C {REPO} fetch origin')
    os.system(f'git -C {REPO} checkout {BRANCH}')
    os.system(f'git -C {REPO} pull --ff-only')
    os.chdir(REPO)
else:
    os.system(f'git clone -b {BRANCH} https://github.com/danort92/Bear-Detector.git')
    os.chdir(REPO)

# Clear cached src.* modules so the freshly-pulled code is always used
import sys
for key in [k for k in sys.modules if k.startswith('src.')]:
    del sys.modules[key]

print(f'✅ Repository ready (branch: {BRANCH}, cwd: {os.getcwd()})')

In [ ]:
# Install dependencies (takes ~2 min on first run)
!pip install -q -r requirements.txt
print('✅ Dependencies installed')

In [ ]:
import sys, torch
sys.path.insert(0, '.')

from src.utils.device import get_device
from src.utils.seed import set_seed

DEVICE = get_device('auto')
set_seed(42)

print(f'🖥  Device  : {DEVICE}')
print(f'🔥 PyTorch : {torch.__version__}')
if DEVICE == 'cuda':
    print(f'🎮 GPU     : {torch.cuda.get_device_name(0)}')
    print(f'   VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2 · Load Configuration

In [ ]:
from src.utils.config import load_merged_config
import yaml

cfg = load_merged_config('config/default.yaml')

# ── Override anything you like here ──────────────────────────
cfg['training']['detection']['epochs']      = 20    # increase for better results
cfg['training']['detection']['batch_size']  = 16
cfg['experiment']['seed']                   = 42
cfg['mlflow']['enabled']                    = False  # set True to use MLflow UI
# ─────────────────────────────────────────────────────────────

print(yaml.dump(cfg, default_flow_style=False))

## 3 · Train YOLOv8 Detection Model

The dataset is the 1,172-image Roboflow bear-face dataset already included in the repo.
Training with `yolov8n` (nano) takes ~5–10 min on a T4 GPU for 20 epochs.

In [ ]:
from src.training.train_detection import DetectionTrainer
from src.training.experiment_tracker import ExperimentTracker
from src.utils.seed import set_seed

set_seed(cfg['experiment']['seed'])

tracker = ExperimentTracker(cfg)

with tracker.start_run() as run:
    run.log_params({
        'architecture' : cfg['model']['detection']['architecture'],
        'epochs'       : cfg['training']['detection']['epochs'],
        'lr'           : cfg['training']['detection']['learning_rate'],
        'seed'         : cfg['experiment']['seed'],
    })
    trainer = DetectionTrainer(cfg)
    output  = trainer.train()

BEST_PT = output['best_weights']
print(f'\n✅ Training done. Best weights → {BEST_PT}')

## 4 · Detect Bears in an Image

**How to use your own image:**
1. Set `USE_UPLOAD = True` in the next cell
2. A file picker will appear — choose any `.jpg` / `.png` from your computer
3. Run the detection cell below

Leave `USE_UPLOAD = False` to pick a random image from the built-in test set.

In [ ]:
from pathlib import Path
import random

USE_UPLOAD = False  # set True to upload your own image instead

if USE_UPLOAD:
    from google.colab import files
    uploaded = files.upload()          # opens the file picker
    SAMPLE_IMG = list(uploaded.keys())[0]
    print(f'Uploaded image: {SAMPLE_IMG}')
else:
    test_images = sorted(Path('Bear detection.v3i.yolov8-obb/test/images').glob('*.jpg'))
    SAMPLE_IMG  = str(random.choice(test_images))
    print(f'Sample image: {SAMPLE_IMG}')

In [ ]:
from src.inference.detector import BearDetector
from IPython.display import display
from PIL import Image
import cv2

detector = BearDetector(
    weights_path    = BEST_PT,
    conf_threshold  = cfg['inference']['detection_conf_threshold'],
    iou_threshold   = cfg['inference']['detection_iou_threshold'],
    device          = DEVICE,
)

result = detector.predict_image(SAMPLE_IMG, annotate=True)

print(f'Detections : {len(result["boxes"])}')
for i, (box, score, label) in enumerate(
        zip(result['boxes'], result['scores'], result['labels'])):
    print(f'  [{i}] {label}  conf={score:.3f}  box={[round(v,1) for v in box]}')

# Display annotated image
annotated_rgb = cv2.cvtColor(result['annotated_image'], cv2.COLOR_BGR2RGB)
display(Image.fromarray(annotated_rgb).resize((640, 480)))

## 5 · Detect + Track Bears in a Video

**Three ways to provide a video:**

| Option | How |
|--------|-----|
| **A — Upload from your computer** | Set `USE_UPLOAD = True` in the next cell and pick a file |
| **B — Use the repo sample video** | Leave `USE_UPLOAD = False`; place a video at `data/sample/bear_sample.mp4` (see README) |
| **C — Synthetic test video** | Automatic fallback — random-noise frames so you can test the pipeline without any video |

The output shows each bear with a **coloured bounding box and a track ID** that stays consistent across frames.

In [ ]:
import os
import numpy as np

USE_UPLOAD = False  # set True to upload your own video via file picker

SAMPLE_VIDEO_PATH = 'data/sample/bear_sample.mp4'  # committed sample video (if present)

if USE_UPLOAD:
    from google.colab import files
    uploaded = files.upload()
    INPUT_VIDEO = list(uploaded.keys())[0]
    print(f'Uploaded video: {INPUT_VIDEO}')
elif os.path.isfile(SAMPLE_VIDEO_PATH):
    INPUT_VIDEO = SAMPLE_VIDEO_PATH
    print(f'Using sample video: {INPUT_VIDEO}')
else:
    # Fallback: create a tiny synthetic test video (random noise, 30 frames)
    INPUT_VIDEO = '/tmp/test_bear_video.mp4'
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(INPUT_VIDEO, fourcc, 10.0, (640, 480))
    for _ in range(30):
        frame = np.random.randint(60, 200, (480, 640, 3), dtype=np.uint8)
        writer.write(frame)
    writer.release()
    print(f'No sample video found at {SAMPLE_VIDEO_PATH} — using synthetic test video.')
    print('Tip: add a real bear video to data/sample/bear_sample.mp4 for meaningful results.')

print(f'\nInput video: {INPUT_VIDEO}')

In [ ]:
from src.tracking.sort_tracker import SORTTracker

OUTPUT_VIDEO = 'outputs/videos/bear_tracked.mp4'

tracker_sort = SORTTracker(
    max_age       = cfg['tracking']['max_age'],
    min_hits      = cfg['tracking']['min_hits'],
    iou_threshold = cfg['tracking']['iou_threshold'],
)

summary = detector.process_video(
    video_path  = INPUT_VIDEO,
    output_path = OUTPUT_VIDEO,
    tracker     = tracker_sort,
)
print(summary)

In [ ]:
# Re-encode to H.264 so the downloaded file is playable in any media player
import os
from google.colab import files

H264_VIDEO = OUTPUT_VIDEO.replace('.mp4', '_h264.mp4')
ret = os.system(f'ffmpeg -y -i "{OUTPUT_VIDEO}" -vcodec libx264 -crf 23 "{H264_VIDEO}" -loglevel error')
if ret == 0:
    print(f'Re-encoded to H.264: {H264_VIDEO}')
    files.download(H264_VIDEO)
else:
    print('ffmpeg re-encode failed, downloading original (may not play in browser)')
    files.download(OUTPUT_VIDEO)

## 6 · Instance Segmentation

Requires a YOLO-seg format dataset. Skip training and load pre-trained COCO weights to demo the inference style.

In [ ]:
from src.segmentation.infer_segmentation import BearSegmentor

# Use COCO-pretrained yolov8n-seg to demonstrate the segmentor interface.
# classes=[21] restricts predictions to COCO class 21 ("bear").
# For bear-specific segmentation replace 'yolov8n-seg.pt' with your trained weights
# and remove the classes filter.
segmentor = BearSegmentor(
    weights_path   = 'yolov8n-seg.pt',   # downloads COCO weights automatically
    conf_threshold = 0.25,
    device         = DEVICE,
    classes        = [21],               # COCO class 21 = "bear"
)

seg_result = segmentor.predict_image(SAMPLE_IMG, annotate=True)

print(f'Instances  : {len(seg_result["boxes"])}')
print(f'Masks      : {len(seg_result["masks"])}')
for label, score in zip(seg_result['labels'], seg_result['scores']):
    print(f'  {label}  conf={score:.3f}')

seg_rgb = cv2.cvtColor(seg_result['annotated_image'], cv2.COLOR_BGR2RGB)
display(Image.fromarray(seg_rgb).resize((640, 480)))

### Run segmentation on a video

The same `INPUT_VIDEO` from Section 5 is used. Each frame gets instance masks overlaid on detected bears.
Re-run the video upload cell above to change the source video.

In [ ]:
# Uncomment and set your dataset path:
# cfg['data']['segmentation']['data_dir'] = 'data/processed/segmentation'
# cfg['training']['segmentation']['epochs'] = 50

# from src.segmentation.train_segmentation import SegmentationTrainer
# seg_trainer = SegmentationTrainer(cfg)
# seg_output  = seg_trainer.train()
# print(seg_output)

print('Uncomment the lines above once you have a YOLO-seg dataset ready.')

In [ ]:
import os, cv2
from pathlib import Path

SEG_OUTPUT_VIDEO = 'outputs/videos/bear_segmented.mp4'
Path(SEG_OUTPUT_VIDEO).parent.mkdir(parents=True, exist_ok=True)

cap = cv2.VideoCapture(INPUT_VIDEO)
fps = cap.get(cv2.CAP_PROP_FPS) or 10.0
w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out    = cv2.VideoWriter(SEG_OUTPUT_VIDEO, fourcc, fps, (w, h))

frame_count = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    result = segmentor.predict_image(frame, annotate=True)
    out.write(result['annotated_image'])
    frame_count += 1

cap.release()
out.release()
print(f'Processed {frame_count} frames → {SEG_OUTPUT_VIDEO}')

# Re-encode to H.264 for browser playback
SEG_H264 = SEG_OUTPUT_VIDEO.replace('.mp4', '_h264.mp4')
ret_code  = os.system(f'ffmpeg -y -i "{SEG_OUTPUT_VIDEO}" -vcodec libx264 -crf 23 "{SEG_H264}" -loglevel error')
if ret_code == 0:
    print(f'Re-encoded to H.264: {SEG_H264}')
    from google.colab import files
    files.download(SEG_H264)
else:
    from google.colab import files
    files.download(SEG_OUTPUT_VIDEO)

## 7 · Evaluation

In [ ]:
# Validate detection model on the test split
from src.training.train_detection import DetectionTrainer
import json

det_trainer = DetectionTrainer(cfg)
metrics = det_trainer.validate(BEST_PT)

print('\n=== Detection Metrics ===')
for k, v in metrics.items():
    print(f'  {k:20s}: {v}')

In [ ]:
# Demo tracking metrics on synthetic data
import numpy as np
from src.tracking.metrics import compute_tracking_metrics

def _make(box_id_list):
    return [np.array([*b, tid]) for b, tid in box_id_list]

gt   = [_make([([10,10,50,50], 1)]), _make([([12,12,52,52], 1)])]
pred = [_make([([11,11,49,49], 1)]), _make([([13,13,51,51], 1)])]

trk_metrics = compute_tracking_metrics(gt, pred, iou_threshold=0.5)

print('\n=== Tracking Metrics (synthetic demo) ===')
for k, v in trk_metrics.items():
    print(f'  {k:20s}: {v}')

In [ ]:
# View the PR curve saved during training
from pathlib import Path
from IPython.display import display
from PIL import Image
import glob

pr_files = list(Path('outputs/metrics').glob('*pr_curve*.png')) + \
           list(Path('outputs/metrics').glob('*precision_recall*.png'))

if pr_files:
    display(Image.open(pr_files[0]))
else:
    print('No PR curve found yet — run evaluate_detection() first.')

## 8 · MLflow Experiment Tracking

**What is MLflow?**
MLflow records every training run — hyperparameters, metrics, and model weights — so you can compare experiments and reproduce results.

**How it works in this project:**

```
Training run
   │
   ├─ log_params()   → records epochs, lr, architecture, seed
   ├─ log_metric()   → records mAP, loss per epoch
   └─ log_artifact() → saves best.pt weight file
```

**Two logging modes (set in Section 2):**

| `cfg['mlflow']['enabled']` | What happens |
|----------------------------|--------------|
| `False` (default in Colab) | Metrics written to `outputs/experiments/bear_detector_metrics.json` |
| `True`                     | Full MLflow server at `outputs/experiments/mlruns/` — browse with the UI below |

**To enable the UI:**
1. Change `cfg['mlflow']['enabled'] = True` in the config cell (Section 2)
2. Re-run training (Section 3)
3. Run the cell below to start the server and get a public URL via ngrok

In [ ]:
import json
from pathlib import Path

# ── Option 1: View logged metrics from JSON (works without MLflow enabled) ──────
log_files = sorted(Path('outputs/experiments').glob('*.json'))
if log_files:
    print('=== Experiment JSON logs ===')
    for f in log_files:
        print(f'\n--- {f.name} ---')
        with open(f) as fh:
            print(json.dumps(json.load(fh), indent=2))
else:
    print('No experiment logs yet — run training first (Section 3).')

# ── Option 2: Launch MLflow UI (requires cfg["mlflow"]["enabled"] = True) ───────
# Uncomment to start the server and expose it via ngrok:
#
# !pip install -q pyngrok
# from pyngrok import ngrok
# !mlflow ui --backend-store-uri outputs/experiments/mlruns --port 5000 &
# public_url = ngrok.connect(5000)
# print(f'MLflow UI → {public_url}')
#
# Then open the printed URL in your browser to browse runs, compare metrics,
# and download logged artifacts (best.pt, PR curves, etc.).

## Tips for Colab

| Tip | How |
|-----|-----|
| Save your trained weights | `from google.colab import files; files.download(BEST_PT)` |
| Use Google Drive | `from google.colab import drive; drive.mount('/content/drive')` |
| Monitor GPU usage | `!nvidia-smi` |
| Check YOLO training live | Open `outputs/models/detection/*/results.csv` |
| Resume training | Set `pretrained_weights` in config to a previous `.pt` file |

---

**GitHub:** https://github.com/danort92/Bear-Detector  
**Author:** Danilo Ortelli
